In [ ]:
dbutils.widgets.text("start_date", "2023-10-02")
dbutils.widgets.text("end_date", "2023-10-05")

start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

In [ ]:
# ── Table configuration ─────────────────────────────────────────────────
# Set USE_SAMPLE_DATA=true/false as an environment variable (or Databricks
# cluster/job env var) to switch between trial views (DD-1167) and full
# production views without changing any code.
# Override catalog names via MARKET_DATA_CATALOG / DERIVED_DATA_CATALOG.
import os as _os

_s = _os.environ.get("USE_SAMPLE_DATA", "").lower()
USE_SAMPLE_DATA = True if _s == "true" else (False if _s == "false" else False)

MARKET_DATA_CATALOG  = _os.environ.get("MARKET_DATA_CATALOG",  "market_data")
DERIVED_DATA_CATALOG = _os.environ.get("DERIVED_DATA_CATALOG", "derived_data")

if USE_SAMPLE_DATA:
    # DD-1167 trial views — 6-month rolling window, no subscription required
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.REF_GLOBAL_PROD"
    ANALYTICS_TBL               = f"{MARKET_DATA_CATALOG}.ANA_GLOBAL_PROD"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.TRDS_GLOBAL_PROD"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.TDSP_GLOBAL_PROD"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.L2_GLOBAL_PROD"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.CBBO_GLOBAL_PROD"
    DAILY_CLASSIFIED_TRADES_TBL = f"{MARKET_DATA_CATALOG}.DCT_GLOBAL_PROD"
else:
    # Full production views
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_reference"
    ANALYTICS_TBL               = f"{DERIVED_DATA_CATALOG}.vw_analytics"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.vw_trades"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.vw_trades_plus"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.vw_l2_data_equity"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.vw_cbbo_equity"
    DAILY_CLASSIFIED_TRADES_TBL = f"{DERIVED_DATA_CATALOG}.vw_daily_classified_trades"

## BMLL Notebook Educational Series
### Finding a Signal in the Noise: Order Book Analysis

### Signal Discovery from Market Microstructure Data

**Objective**: Discover alpha-generating trading signals by analyzing order book dynamics.
The fundamental hypothesis is that **price movements are predictable from order book characteristics**.

**Why Order Book Data?**
- Captures market microstructure (limit orders, cancellations, executions)
- Reveals trader intent before prices move
- Enables early detection of momentum and reversals
- Shows supply/demand imbalances at different price levels

**Signal Types** that can be extracted from Level 2/3 order book:
- **Imbalance signals**: Significant bid/ask size differences predict price movement
- **Depth signals**: Shallow book depth often precedes large moves
- **Order flow signals**: Clustering of orders at certain levels indicates support/resistance
- **Momentum signals**: Rate of change in best bid/ask often leads price movement

This notebook uses statistical regression to measure correlations between order book metrics
and subsequent price changes (alpha) across the S&P 500 universe in Databricks.

## Import Libraries

In [ ]:
try:
    import statsmodels
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels"])

import pytz
import pandas as pd
import statsmodels.api as sm

from datetime import datetime, timedelta

## Phase 1: Data Retrieval and Preparation

This section gathers Level 3 order book data (10 levels deep) and preprocesses it for analysis.
The process:
1. **Universe definition**: Select S&P 500 securities traded on US venues
2. **Reference mapping**: Map tickers to BMLL Listing IDs
3. **Data fetching**: Retrieve Level 2/3 order book snapshots at regular intervals
4. **Feature engineering**: Extract imbalance and depth metrics from order book

Level 2 data provides bid/ask prices and quantities at up to 10 price levels.
Level 3 data shows individual orders (more granular but more computation-intensive).

### Data Collection: Level 3 Order Book

### Select Universe (Top 500 stocks)

### Get ListingIds (BMLL Ticker-MIC ids that map to a ticker and venue)

In [ ]:
# Define list of US market MIC codes to filter by
us_mics = ['XNAS', 'XBOS', 'XPSX', 'XNYS', 'XASE', 'ARCX', 'BATY',
           'BATS', 'EDGA', 'EDGX', 'XCIS', 'XCHI', 'IEXG', 'MEMX', 'EPRL']

In [ ]:
mic_list = ", ".join(f"'{mic}'" for mic in us_mics)
ref_sql = f"""
    SELECT DISTINCT TICKER AS Ticker
    FROM {REFERENCE_TBL}
    WHERE IS_ALIVE = TRUE
      AND IS_PRIMARY = TRUE
      AND MIC IN ({mic_list})
    LIMIT 500
"""
universe = spark.sql(ref_sql).toPandas()
universe_symbols = universe["Ticker"].tolist()

ref_lid_sql = f"""
    SELECT LISTING_ID AS ListingId
    FROM {REFERENCE_TBL}
    WHERE IS_ALIVE = TRUE
      AND IS_PRIMARY = TRUE
      AND MIC IN ({mic_list})
      AND TICKER IN ({", ".join(f"'{t}'" for t in universe_symbols)})
"""
universe_lids = spark.sql(ref_lid_sql).toPandas()["ListingId"].tolist()

### Time Period and other variables

In [ ]:
start_date = "2023-10-02"
end_date = "2023-10-05"
freq = 60 ## interval size

# Convert start_date and end_date strings to datetime objects
start_date_obj = datetime.strptime(start_date, "%Y-%m-%d")
end_date_obj = datetime.strptime(end_date, "%Y-%m-%d")

# Generate a list of dates between start_date and end_date (inclusive)
dates = [start_date_obj + timedelta(days=x) for x in range((end_date_obj - start_date_obj).days + 1)]

# Convert date objects back to strings
dates = [date.strftime("%Y-%m-%d") for date in dates]

In [ ]:
start_time = "09:30"
end_time = "16:00"

### Collect L3 Data

In [ ]:
def _utc_bounds(date, start_time, end_time):
    ts_start = pd.Timestamp(f"{date} {start_time}").tz_localize("US/Eastern").tz_convert("UTC").tz_localize(None)
    ts_end = pd.Timestamp(f"{date} {end_time}").tz_localize("US/Eastern").tz_convert("UTC").tz_localize(None)
    return ts_start, ts_end

In [ ]:
def get_L2_data(lid, date, start_time, end_time, freq=60, depth=10):
    ts_start, ts_end = _utc_bounds(date, start_time, end_time)
    level_cols = []
    for i in range(1, depth + 1):
        level_cols.extend([
            f"BID_PRICE{i}", f"BID_QUANTITY{i}",
            f"ASK_PRICE{i}", f"ASK_QUANTITY{i}",
        ])
    cols_sql = ", ".join(["EVENT_TIMESTAMP", "BMLL_OBJECT_ID AS LISTING_ID"] + level_cols)
    sql = f"""
        SELECT {cols_sql}
        FROM `market_data`.`vw_l2_data_equity`
        WHERE BMLL_OBJECT_ID = {lid}
          AND TRADE_DATE = '{date}'
          AND EVENT_TIMESTAMP BETWEEN '{ts_start}' AND '{ts_end}'
        ORDER BY EVENT_TIMESTAMP
    """
    df = spark.sql(sql).toPandas()
    if df.empty:
        return pd.DataFrame(columns=["timestamp", "lid", "side", "price", "size"])
    df["EVENT_TIMESTAMP"] = pd.to_datetime(df["EVENT_TIMESTAMP"])
    df = df.sort_values("EVENT_TIMESTAMP")
    df = (
        df.groupby("LISTING_ID", group_keys=False)
        .resample(f"{freq}S", on="EVENT_TIMESTAMP")
        .last()
        .reset_index()
    )

    records = []
    for _, row in df.iterrows():
        ts = row["EVENT_TIMESTAMP"]
        lid_val = row["LISTING_ID"]
        for i in range(1, depth + 1):
            bid_p = row.get(f"BID_PRICE{i}")
            bid_q = row.get(f"BID_QUANTITY{i}")
            ask_p = row.get(f"ASK_PRICE{i}")
            ask_q = row.get(f"ASK_QUANTITY{i}")
            if pd.notna(bid_p) and pd.notna(bid_q):
                records.append((ts, lid_val, 1, bid_p, bid_q))
            if pd.notna(ask_p) and pd.notna(ask_q):
                records.append((ts, lid_val, 2, ask_p, ask_q))
    return pd.DataFrame(records, columns=["timestamp", "lid", "side", "price", "size"])

In [ ]:
l3 = get_L2_data(322753, "2023-10-02", start_time, end_time, freq=freq, depth=10)

## Phase 2: Feature Engineering and Signal Extraction

### Signal Metrics for Predictiveness

We extract three key order book imbalance metrics at fixed time intervals. These ratios measure
buy-side vs sell-side characteristics, which often precede price movements:

**1. Notional Imbalance Ratio**
- Ratio of total buy-side volume to sell-side volume
- **Interpretation**: Values > 1 suggest more buying pressure (bullish)
- **Signal**: Large ratios often precede upward price movements

**2. Average Size Ratio**
- Ratio of average buy order size to average sell order size
- **Interpretation**: Larger buy orders suggest institutional buying
- **Signal**: Disparity in order sizes can indicate informed trading

**3. Order Count Ratio**
- Ratio of number of buy orders to sell orders
- **Interpretation**: More buy orders may indicate bid-side support
- **Signal**: Order count alone has weaker predictive power but useful in combination

**Why These Metrics?**
The hypothesis is that market microstructure (how orders are placed, not just their prices)
contains information about future price movements. By analyzing these ratios at regular intervals,
we can identify patterns that precede directional moves.

### Calculate Imbalance Ratios at Each Time Snapshot

In [ ]:
# Step 1: Separate buy-side (side=1) and sell-side (side=2) orders
side1_data = l3[l3['side'] == 1]
side2_data = l3[l3['side'] == 2]

# Step 2: Group by timestamp and security (lid) to analyze book state at each snapshot
grouped_side1 = side1_data.groupby(['timestamp', 'lid'])
grouped_side2 = side2_data.groupby(['timestamp', 'lid'])

# Step 3: Calculate three aggregate metrics per side
# - sum: total notional volume (sum of all sizes)
# - mean: average order size per level
# - size: order count
sum_notional_side1 = grouped_side1['size'].sum()
avg_size_side1 = grouped_side1['size'].mean()
count_side1 = grouped_side1.size()

sum_notional_side2 = grouped_side2['size'].sum()
avg_size_side2 = grouped_side2['size'].mean()
count_side2 = grouped_side2.size()

# Step 4: Compute ratios (buy-side / sell-side)
# Ratios > 1 indicate buy-side dominance
sum_notional_ratio = sum_notional_side1 / sum_notional_side2
avg_size_ratio = avg_size_side1 / avg_size_side2
count_ratio = count_side1 / count_side2

# Step 5: Consolidate all ratios into a single DataFrame for regression
ratios_df = pd.DataFrame({
    'timestamp': sum_notional_ratio.index.get_level_values('timestamp'),
    'lid': sum_notional_ratio.index.get_level_values('lid'),
    'sum_notional_ratio': sum_notional_ratio.values,
    'avg_size_ratio': avg_size_ratio.values,
    'count_ratio': count_ratio.values
})

ratios_df

### Calculate CBBO at Each Snapshot

We compute the consolidated best bid/offer (CBBO) by finding:
- **Best bid**: Maximum bid price across all levels (buy-side strength)
- **Best ask**: Minimum ask price across all levels (sell-side strength)
- **Mid price**: Average of best bid and ask (fair value benchmark)

The mid price serves as the dependent variable in our regression - we want to know if
the order book imbalance ratios can predict future mid prices.

In [ ]:
# Step 1: Ensure timestamps are in datetime format
l3['timestamp'] = pd.to_datetime(l3['timestamp'])

# Step 2: Separate buy and sell side orders
side1_prices = l3[l3['side'] == 1]  # Buy side (bids)
side2_prices = l3[l3['side'] == 2]  # Sell side (asks)

# Step 3: Find best bid (max buy price) and best ask (min sell price) at each snapshot
# These represent the CBBO (consolidated best bid/offer)
side1_max_prices = side1_prices.groupby(['timestamp', 'lid'])['price'].max().reset_index()
side2_min_prices = side2_prices.groupby(['timestamp', 'lid'])['price'].min().reset_index()

# Step 4: Merge bid and ask prices
final_prices = pd.merge(side1_max_prices, side2_min_prices, on=['timestamp', 'lid'], suffixes=('_side1', '_side2'))

# Step 5: Calculate mid price as the average of bid and ask
final_prices['mid_price'] = (final_prices['price_side2'] + final_prices['price_side1']) / 2

# Step 6: Merge imbalance ratios with prices for regression analysis
ratio_mid = ratios_df.merge(final_prices)
ratio_mid

## Phase 3: Statistical Analysis and Hypothesis Testing

Now we test whether the order book imbalance metrics have predictive power for price movements.
The analysis includes:
1. **Correlation analysis**: Measure linear relationship between signals and prices
2. **Regression modeling**: Quantify the impact of each signal
3. **Statistical significance**: Determine if observed relationships are meaningful

### Correlation Analysis: Signal Strength

We first compute simple correlations between each imbalance metric and price.
Strong correlations suggest the metric has predictive value, though correlation
alone does not establish causality.

In [ ]:
# Step 1: Calculate pairwise correlation between each imbalance metric and mid price
# Correlation ranges from -1 (inverse) to +1 (positive relationship)
# A value close to 0 suggests no linear relationship
correlation = ratio_mid['sum_notional_ratio'].corr(ratio_mid['mid_price'])
print("Correlation between 'notional_ratio' and 'mid_price':", correlation)

In [ ]:
# ## Regression Analysis: Quantifying Predictive Power
#
# ### Model Specification
#
# We use Ordinary Least Squares (OLS) regression to estimate the relationship:
# **mid_price = β₀ + β₁(notional_ratio) + β₂(size_ratio) + β₃(count_ratio) + ε**
#
# **Interpretation of coefficients:**
# - **β₀ (intercept)**: Base price level
# - **β₁-β₃**: Change in price for a unit change in each ratio (holding others constant)
# - **P-values**: Probability that the coefficient is zero (< 0.05 suggests significance)
# - **R²**: Proportion of price variance explained by the model

In [ ]:
# Step 1: Define dependent variable (what we want to predict)
y = ratio_mid['mid_price']

# Step 2: Define independent variables (what we use to predict)
X = ratio_mid[['sum_notional_ratio', 'avg_size_ratio', 'count_ratio']]

# Step 3: Add intercept term (constant) for regression
X = sm.add_constant(X)

# Step 4: Fit OLS regression model
model = sm.OLS(y, X).fit()

# Step 5: Display comprehensive regression results
print(model.summary())

In [ ]:
# If it's not already, make sure 'lid' is of type category for regression analysis

# Assuming ratio_mid is your DataFrame
# Convert 'lid' to categorical type
ratio_mid['lid'] = ratio_mid['lid'].astype('category')

# Define independent variables
X = ratio_mid[['sum_notional_ratio', 'avg_size_ratio', 'count_ratio']]

# Add constant term to the independent variables
X = sm.add_constant(X)

# Define dependent variable
y = ratio_mid['mid_price']

# Perform regression for each lid
results = {}
for lid, group in ratio_mid.groupby('lid'):
    lid_str = str(lid)  # Convert lid to string for use as dictionary key
    X_group = X[ratio_mid['lid'] == lid]  # Subset X for current lid
    y_group = y[ratio_mid['lid'] == lid]  # Subset y for current lid
    
    # Fit OLS regression model
    model = sm.OLS(y_group, X_group)
    result = model.fit()
    results[lid_str] = result

# Display regression results
for lid, result in results.items():
    print(f"Regression results for lid {lid}:")
    print(result.summary())